In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import col, to_date

# 1. Define the raw data
data = [
    (101, "Amit", 50000, "2026-01-01"),
    (101, "Amit", 55000, "2026-02-01"),
    (102, "Neha", 60000, "2026-01-01"),
    (103, "Rahul", 70000, "2026-01-01"),
    (103, "Rahul", 75000, "2026-03-01")
]

# 2. Define the schema
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("updated_at", StringType(), True)
])

# 3. Create the DataFrame and convert the date string to an actual Date type
df = spark.createDataFrame(data, schema) \
    .withColumn("updated_at", to_date(col("updated_at"), "yyyy-MM-dd"))

# 4. Display the DataFrame
df.display()

In [0]:
df.createOrReplaceTempView('employee')

In [0]:
%sql
with ranked_emp as(
    select
    *,
    ROW_NUMBER() OVER( PARTITION BY emp_id ORDER BY updated_at DESC) as rn 
    FROM employee
    )
    select * from ranked_emp
    where rn=1;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

win_spec = Window.partitionBy('emp_id').orderBy(F.col('updated_at').desc())

dedup_df = df.withColumn('rn',
                         F.row_number().over(win_spec)) \
                             .filter(F.col('rn')==1)\
                                 .drop('rn')


In [0]:
dedup_df.show()

# Scenarios based


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, to_date

# ==========================================
# 1. Table 1: Store Visits
# ==========================================
visits_data = [
    ("V101", "C001", "2026-10-15"),
    ("V102", "C002", "2026-10-15"),
    ("V103", "C003", "2026-10-15"),
    ("V104", "C004", "2026-10-15"),
    ("V105", "C005", "2026-10-15"),
    ("V106", "C002", "2026-10-16")
]

visits_schema = StructType([
    StructField("visit_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("visit_date", StringType(), True)
])

# Create DataFrame and convert visit_date string to Date type
df_visits = spark.createDataFrame(visits_data, visits_schema) \
    .withColumn("visit_date", to_date(col("visit_date"), "yyyy-MM-dd"))

# ==========================================
# 2. Table 2: Purchases
# ==========================================
purchases_data = [
    ("P101", "V101", "C001", 1500),
    ("P102", "V103", "C003", 800),
    ("P103", "V105", "C005", 2000),
    ("P104", "V106", "C002", 500)
]

purchases_schema = StructType([
    StructField("purchase_id", StringType(), True),
    StructField("visit_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", IntegerType(), True)
])

df_purchases = spark.createDataFrame(purchases_data, purchases_schema)

# Display both DataFrames in Databricks
display(df_visits)
display(df_purchases)

In [0]:
df_visits.join(df_purchases, "customer_id", 'leftanti').show()
df_visits.join(df_purchases, "customer_id", 'right').show()
